# 📊 Praktikum Machine Learning: Regresi & Klasifikasi KNN

**Materi:**
1. Regresi Linear Sederhana
2. Regresi Linear Berganda
3. Regresi Logistik
4. Klasifikasi KNN (data sendiri)
5. Klasifikasi KNN (dataset Kaggle / UCI)

---

## 🔧 Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('✅ Semua library berhasil diimport!')

---
# 📌 BAGIAN 1 — Regresi Linear Sederhana

**Konsep:** Regresi linear sederhana digunakan untuk memodelkan hubungan antara SATU variabel independen (X) dan SATU variabel dependen (Y). Persamaannya: `Y = a + bX`

**Studi Kasus:** Apakah jam belajar mempengaruhi nilai ujian mahasiswa?

In [ ]:
# ── DATA BUATAN SENDIRI ──────────────────────────────────────────────
# Skenario: jam belajar per minggu vs nilai ujian (skala 0–100)
np.random.seed(7)

jam_belajar = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                         2, 4, 6, 3, 7, 5, 8, 1, 9, 6])
# Nilai ujian = 40 + 5×jam + noise kecil
nilai_ujian  = 40 + 5 * jam_belajar + np.random.normal(0, 4, size=len(jam_belajar))
nilai_ujian  = np.clip(nilai_ujian, 0, 100)   # pastikan 0–100

df_rl_simple = pd.DataFrame({
    'jam_belajar': jam_belajar,
    'nilai_ujian' : nilai_ujian
})

print(df_rl_simple.head(10))
print(f"\nJumlah data: {len(df_rl_simple)} baris")

In [ ]:
# ── MODELING ──────────────────────────────────────────────────────────
X_simple = df_rl_simple[['jam_belajar']]
y_simple  = df_rl_simple['nilai_ujian']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.25, random_state=42
)

model_simple = LinearRegression()
model_simple.fit(X_train_s, y_train_s)

y_pred_s = model_simple.predict(X_test_s)

print(f"Intercept (a)  : {model_simple.intercept_:.2f}")
print(f"Koefisien (b)  : {model_simple.coef_[0]:.2f}")
print(f"Persamaan      : Nilai = {model_simple.intercept_:.2f} + {model_simple.coef_[0]:.2f} × Jam")
print(f"\nR² Score       : {r2_score(y_test_s, y_pred_s):.4f}")
print(f"RMSE           : {np.sqrt(mean_squared_error(y_test_s, y_pred_s)):.4f}")

In [ ]:
# ── VISUALISASI ───────────────────────────────────────────────────────
x_line = np.linspace(jam_belajar.min(), jam_belajar.max(), 100).reshape(-1,1)
y_line = model_simple.predict(x_line)

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(X_train_s, y_train_s, color='steelblue', label='Data Latih', zorder=3)
ax.scatter(X_test_s,  y_test_s,  color='orange',    label='Data Uji',   zorder=3, marker='D')
ax.plot(x_line, y_line, color='crimson', linewidth=2, label='Garis Regresi')
ax.set_xlabel('Jam Belajar per Minggu')
ax.set_ylabel('Nilai Ujian')
ax.set_title('Regresi Linear Sederhana: Jam Belajar vs Nilai Ujian')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Linear Sederhana

- **Persamaan model** yang diperoleh adalah `Nilai ≈ a + b × Jam_Belajar`. Nilai intercept menunjukkan estimasi nilai ujian ketika mahasiswa tidak belajar sama sekali.
- **Koefisien b** menunjukkan bahwa setiap tambahan 1 jam belajar per minggu diperkirakan meningkatkan nilai ujian sebesar nilai b poin.
- **R² ≈ 0.9+** berarti sekitar 90% variasi nilai ujian dapat dijelaskan oleh jumlah jam belajar — hubungan yang cukup kuat.
- **RMSE** yang kecil mengindikasikan prediksi model cukup dekat dengan nilai aktual.
- **Kesimpulan:** Jam belajar memiliki pengaruh positif yang signifikan terhadap nilai ujian mahasiswa.

---
# 📌 BAGIAN 2 — Regresi Linear Berganda

**Konsep:** Regresi linear berganda menggunakan LEBIH DARI SATU variabel independen untuk memprediksi variabel dependen. Persamaan: `Y = a + b1X1 + b2X2 + ... + bnXn`

**Studi Kasus:** Memprediksi harga rumah berdasarkan luas bangunan, jumlah kamar, dan jarak ke pusat kota.

In [ ]:
# ── DATA BUATAN SENDIRI ──────────────────────────────────────────────
np.random.seed(21)
n = 80

luas_bangunan  = np.random.randint(36, 200, n)          # m²
jumlah_kamar   = np.random.randint(1, 6, n)             # 1–5 kamar
jarak_kota_km  = np.round(np.random.uniform(1, 30, n), 1)  # km

# Harga (juta Rp) = 150 + 3×luas + 20×kamar − 5×jarak + noise
harga = (150
         + 3   * luas_bangunan
         + 20  * jumlah_kamar
         - 5   * jarak_kota_km
         + np.random.normal(0, 25, n))

df_multi = pd.DataFrame({
    'luas_m2'    : luas_bangunan,
    'kamar'      : jumlah_kamar,
    'jarak_km'   : jarak_kota_km,
    'harga_juta' : np.round(harga, 2)
})

print(df_multi.head())
print(f"\nStatistik deskriptif:")
print(df_multi.describe().round(2))

In [ ]:
# ── MODELING ──────────────────────────────────────────────────────────
X_multi = df_multi[['luas_m2', 'kamar', 'jarak_km']]
y_multi  = df_multi['harga_juta']

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_multi, y_multi, test_size=0.25, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_tr_m, y_tr_m)
y_pred_m = model_multi.predict(X_te_m)

print("Koefisien tiap fitur:")
for feat, coef in zip(X_multi.columns, model_multi.coef_):
    print(f"  {feat:12s}: {coef:.4f}")
print(f"\nIntercept   : {model_multi.intercept_:.4f}")
print(f"R² Score    : {r2_score(y_te_m, y_pred_m):.4f}")
print(f"RMSE        : {np.sqrt(mean_squared_error(y_te_m, y_pred_m)):.4f} juta Rp")

In [ ]:
# ── VISUALISASI: Aktual vs Prediksi + Residual ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Aktual vs Prediksi
axes[0].scatter(y_te_m, y_pred_m, color='teal', alpha=0.7, edgecolors='k', linewidths=0.4)
lims = [min(y_te_m.min(), y_pred_m.min()), max(y_te_m.max(), y_pred_m.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Prediksi Sempurna')
axes[0].set_xlabel('Harga Aktual (juta Rp)')
axes[0].set_ylabel('Harga Prediksi (juta Rp)')
axes[0].set_title('Aktual vs Prediksi')
axes[0].legend()

# Plot 2: Distribusi Residual
residuals = y_te_m.values - y_pred_m
axes[1].hist(residuals, bins=10, color='salmon', edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (juta Rp)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Distribusi Residual')

plt.suptitle('Regresi Linear Berganda — Prediksi Harga Rumah', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Linear Berganda

- **Koefisien luas_m2** bernilai positif → semakin luas rumah, harga semakin tinggi. Setiap tambahan 1 m² meningkatkan harga sekitar 3 juta Rp.
- **Koefisien kamar** bernilai positif → kamar lebih banyak = harga lebih mahal, naik ~20 juta per kamar tambahan.
- **Koefisien jarak_km** bernilai negatif → semakin jauh dari pusat kota, harga turun sekitar 5 juta per km. Ini logis karena aksesibilitas mempengaruhi harga properti.
- **R² ≈ 0.95+** menunjukkan model sangat baik menjelaskan variasi harga rumah.
- **Distribusi residual** mendekati normal dan terpusat di 0, mengindikasikan model tidak bias.
- **Kesimpulan:** Ketiga fitur memiliki kontribusi signifikan dalam menentukan harga rumah. Luas bangunan adalah faktor dominan.

---
# 📌 BAGIAN 3 — Regresi Logistik

**Konsep:** Regresi logistik digunakan untuk **klasifikasi biner** — memprediksi probabilitas suatu kejadian (0 atau 1). Menggunakan fungsi sigmoid untuk mengubah output linear menjadi probabilitas.

**Studi Kasus:** Memprediksi apakah seorang pasien terkena diabetes (1) atau tidak (0) berdasarkan kadar gula darah, BMI, dan usia.

In [ ]:
# ── DATA BUATAN SENDIRI ──────────────────────────────────────────────
np.random.seed(99)
n = 120

gula_darah = np.random.randint(70, 200, n)      # mg/dL
bmi        = np.round(np.random.uniform(17, 40, n), 1)
usia       = np.random.randint(20, 70, n)

# Logit: semakin tinggi gula, bmi, usia → lebih mungkin diabetes
logit = -8 + 0.04*gula_darah + 0.12*bmi + 0.05*usia
prob  = 1 / (1 + np.exp(-logit))
diabetes = (prob > 0.5).astype(int)

df_logit = pd.DataFrame({
    'gula_darah': gula_darah,
    'bmi'       : bmi,
    'usia'      : usia,
    'diabetes'  : diabetes
})

print(df_logit.head(10))
print(f"\nDistribusi kelas:")
print(df_logit['diabetes'].value_counts())

In [ ]:
# ── MODELING ──────────────────────────────────────────────────────────
X_log = df_logit[['gula_darah', 'bmi', 'usia']]
y_log = df_logit['diabetes']

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_log, y_log, test_size=0.25, random_state=42, stratify=y_log
)

# Normalisasi sangat disarankan untuk regresi logistik
scaler_l = StandardScaler()
X_tr_l_sc = scaler_l.fit_transform(X_tr_l)
X_te_l_sc = scaler_l.transform(X_te_l)

model_logit = LogisticRegression(max_iter=300, random_state=42)
model_logit.fit(X_tr_l_sc, y_tr_l)
y_pred_l = model_logit.predict(X_te_l_sc)

print("Accuracy  :", accuracy_score(y_te_l, y_pred_l))
print("\nClassification Report:")
print(classification_report(y_te_l, y_pred_l, target_names=['Tidak Diabetes','Diabetes']))

In [ ]:
# ── VISUALISASI: Confusion Matrix ─────────────────────────────────────
cm = confusion_matrix(y_te_l, y_pred_l)

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Tidak','Pred: Diabetes'],
            yticklabels=['Aktual: Tidak','Aktual: Diabetes'])
ax.set_title('Confusion Matrix — Regresi Logistik Diabetes')
ax.set_ylabel('Aktual')
ax.set_xlabel('Prediksi')
plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Logistik

- **Regresi logistik** menghasilkan output probabilitas (0–1) menggunakan fungsi sigmoid, lalu dikonversi ke kelas biner dengan threshold 0.5.
- Model mampu memisahkan pasien diabetes dan non-diabetes dengan akurasi yang cukup tinggi.
- **Confusion matrix** menunjukkan:
  - True Negative (TN): benar diprediksi tidak diabetes
  - True Positive (TP): benar diprediksi diabetes
  - False Positive (FP): salah didiagnosis diabetes (Type I error)
  - False Negative (FN): pasien diabetes lolos deteksi (Type II error — berbahaya di medis!)
- **Precision** mengukur ketepatan prediksi positif; **Recall** mengukur seberapa banyak kasus positif berhasil terdeteksi.
- **Kesimpulan:** Ketiga variabel (gula darah, BMI, usia) berkontribusi dalam mendeteksi risiko diabetes. Dalam konteks medis, recall untuk kelas diabetes harus dioptimalkan untuk mengurangi false negative.

---
# 📌 BAGIAN 4 — Klasifikasi KNN (Data Sendiri)

**Konsep:** K-Nearest Neighbors (KNN) adalah algoritma klasifikasi yang mengklasifikasikan data baru berdasarkan mayoritas kelas dari K tetangga terdekatnya. Tidak ada 'pelatihan' eksplisit — model menyimpan semua data latih.

**Studi Kasus:** Mengklasifikasikan jenis buah berdasarkan berat (gram) dan kandungan air (%).

In [ ]:
# ── DATA BUATAN SENDIRI ──────────────────────────────────────────────
# 3 kelas buah: Jeruk (0), Apel (1), Melon (2)
np.random.seed(55)

def buat_buah(berat_mean, air_mean, n, label):
    berat = np.random.normal(berat_mean, 15, n)
    air   = np.random.normal(air_mean,  3,  n)
    return pd.DataFrame({'berat_g': berat.round(1), 'kandungan_air': air.round(1), 'buah': label})

df_jeruk = buat_buah(150, 86, 40, 'Jeruk')
df_apel  = buat_buah(200, 84, 40, 'Apel')
df_melon = buat_buah(800, 92, 40, 'Melon')

df_buah = pd.concat([df_jeruk, df_apel, df_melon], ignore_index=True)
df_buah = df_buah.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_buah.head(10))
print(f"\nDistribusi kelas:")
print(df_buah['buah'].value_counts())

In [ ]:
# ── MODELING KNN ──────────────────────────────────────────────────────
X_buah = df_buah[['berat_g', 'kandungan_air']]
y_buah = df_buah['buah']

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_buah, y_buah, test_size=0.25, random_state=42, stratify=y_buah
)

scaler_b = StandardScaler()
X_tr_b_sc = scaler_b.fit_transform(X_tr_b)
X_te_b_sc = scaler_b.transform(X_te_b)

# Coba beberapa nilai K
hasil_k = {}
for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_b_sc, y_tr_b)
    acc = accuracy_score(y_te_b, knn.predict(X_te_b_sc))
    hasil_k[k] = acc

k_terbaik = max(hasil_k, key=hasil_k.get)
print(f"K terbaik : {k_terbaik} (Accuracy = {hasil_k[k_terbaik]:.4f})")

# Train ulang dengan K terbaik
knn_buah = KNeighborsClassifier(n_neighbors=k_terbaik)
knn_buah.fit(X_tr_b_sc, y_tr_b)
y_pred_b = knn_buah.predict(X_te_b_sc)

print("\nClassification Report:")
print(classification_report(y_te_b, y_pred_b))

In [ ]:
# ── VISUALISASI ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy vs K
axes[0].plot(list(hasil_k.keys()), list(hasil_k.values()), marker='o', color='purple')
axes[0].axvline(k_terbaik, color='red', linestyle='--', label=f'K={k_terbaik} (terbaik)')
axes[0].set_xlabel('Nilai K')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Nilai K')
axes[0].legend()
axes[0].set_xticks(range(1,16))

# Plot 2: Scatter data asli
warna = {'Jeruk': 'orange', 'Apel': 'green', 'Melon': 'yellowgreen'}
for kelas, warna_k in warna.items():
    mask = df_buah['buah'] == kelas
    axes[1].scatter(df_buah[mask]['berat_g'], df_buah[mask]['kandungan_air'],
                    c=warna_k, label=kelas, alpha=0.7, edgecolors='k', linewidths=0.3, s=60)
axes[1].set_xlabel('Berat (gram)')
axes[1].set_ylabel('Kandungan Air (%)')
axes[1].set_title('Sebaran Data Buah')
axes[1].legend()

plt.suptitle('KNN — Klasifikasi Jenis Buah', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Analisis KNN (Data Sendiri)

- **KNN** bekerja dengan prinsip 'kemiripan' — data baru diklasifikasikan berdasarkan mayoritas kelas dari K tetangga terdekat.
- **Normalisasi (StandardScaler)** sangat penting karena KNN sensitif terhadap skala fitur. Berat (gram) dan kandungan air (%) memiliki satuan berbeda, sehingga tanpa normalisasi, berat akan mendominasi perhitungan jarak.
- Dari grafik **Accuracy vs K**, terlihat bahwa K terlalu kecil (misal K=1) rentan overfitting (terlalu hafal data latih), sementara K terlalu besar menyebabkan underfitting.
- Data buah memiliki **cluster yang terpisah jelas** — terutama Melon (berat jauh lebih besar), sehingga KNN bekerja sangat baik pada kasus ini.
- **Kesimpulan:** KNN cocok untuk kasus dengan batas kelas yang cukup jelas. Pemilihan nilai K yang tepat sangat krusial dan sebaiknya dilakukan dengan eksperimen atau cross-validation.

---
# 📌 BAGIAN 5 — KNN dengan Dataset dari Kaggle (UCI Heart Disease)

**Dataset:** Heart Disease UCI (tersedia di Kaggle dan langsung dari UCI Repository)

**Target:** Memprediksi apakah pasien memiliki penyakit jantung (1) atau tidak (0)

Dataset ini mengandung 13 fitur medis seperti usia, tekanan darah, kolesterol, dll.

In [ ]:
# ── LOAD DATASET ─────────────────────────────────────────────────────
# Dataset: Heart Disease UCI dari Kaggle
# Cara unduh dari Kaggle:
#   1. Buat akun Kaggle → https://www.kaggle.com/datasets/ronitf/heart-disease-uci
#   2. Download heart.csv → upload ke Colab (Files > Upload)
#   3. Ganti path di bawah jika perlu
#
# Alternatif: load langsung dari URL publik UCI
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heartdisease.csv'

try:
    df_heart = pd.read_csv(url)
    print('✅ Dataset berhasil dimuat dari URL!')
except:
    # Fallback: simulasi struktur dataset heart disease
    print('⚠️ Tidak dapat mengakses URL. Menggunakan data simulasi dengan struktur sama...')
    np.random.seed(13)
    n = 303
    df_heart = pd.DataFrame({
        'age'     : np.random.randint(29, 77, n),
        'sex'     : np.random.randint(0, 2, n),
        'cp'      : np.random.randint(0, 4, n),
        'trestbps': np.random.randint(94, 200, n),
        'chol'    : np.random.randint(126, 564, n),
        'fbs'     : np.random.randint(0, 2, n),
        'restecg' : np.random.randint(0, 3, n),
        'thalach' : np.random.randint(71, 202, n),
        'exang'   : np.random.randint(0, 2, n),
        'oldpeak' : np.round(np.random.uniform(0, 6.2, n), 1),
        'slope'   : np.random.randint(0, 3, n),
        'ca'      : np.random.randint(0, 5, n),
        'thal'    : np.random.randint(0, 4, n),
        'target'  : np.random.randint(0, 2, n)
    })

print(f"\nShape dataset: {df_heart.shape}")
print(df_heart.head())

In [ ]:
# ── EDA SINGKAT ───────────────────────────────────────────────────────
print("Info dataset:")
print(df_heart.info())
print("\nNilai null per kolom:")
print(df_heart.isnull().sum())
print("\nDistribusi target:")
print(df_heart['target'].value_counts())

In [ ]:
# ── VISUALISASI EDA ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribusi target
df_heart['target'].value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue','salmon'], edgecolor='black', rot=0)
axes[0].set_title('Distribusi Kelas Target')
axes[0].set_xlabel('0 = Tidak Sakit, 1 = Sakit Jantung')
axes[0].set_ylabel('Jumlah Pasien')

# Distribusi usia
df_heart['age'].hist(ax=axes[1], bins=20, color='teal', edgecolor='black')
axes[1].set_title('Distribusi Usia Pasien')
axes[1].set_xlabel('Usia')

# Korelasi heatmap (top features)
top_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'target']
existing = [c for c in top_cols if c in df_heart.columns]
sns.heatmap(df_heart[existing].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=axes[2], linewidths=0.5)
axes[2].set_title('Korelasi Fitur Numerik')

plt.suptitle('Eksplorasi Dataset Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PREPROCESSING & MODELING KNN ─────────────────────────────────────
target_col = 'target'
X_heart = df_heart.drop(columns=[target_col])
y_heart = df_heart[target_col]

# Handle missing values jika ada
X_heart = X_heart.fillna(X_heart.median())

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_heart, y_heart, test_size=0.2, random_state=42, stratify=y_heart
)

scaler_h = StandardScaler()
X_tr_h_sc = scaler_h.fit_transform(X_tr_h)
X_te_h_sc = scaler_h.transform(X_te_h)

# Cari K optimal
hasil_k_h = {}
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_tr_h_sc, y_tr_h)
    hasil_k_h[k] = accuracy_score(y_te_h, knn.predict(X_te_h_sc))

k_opt = max(hasil_k_h, key=hasil_k_h.get)
print(f"K optimal : {k_opt}")
print(f"Accuracy  : {hasil_k_h[k_opt]:.4f}")

# Model final
knn_heart = KNeighborsClassifier(n_neighbors=k_opt, metric='euclidean')
knn_heart.fit(X_tr_h_sc, y_tr_h)
y_pred_h = knn_heart.predict(X_te_h_sc)

print("\nClassification Report:")
print(classification_report(y_te_h, y_pred_h, target_names=['Tidak Sakit','Sakit Jantung']))

In [ ]:
# ── VISUALISASI HASIL KNN ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Accuracy vs K
axes[0].plot(list(hasil_k_h.keys()), list(hasil_k_h.values()), marker='s', color='darkblue')
axes[0].axvline(k_opt, color='crimson', linestyle='--', label=f'K={k_opt} optimal')
axes[0].set_xlabel('Nilai K')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Nilai K (Heart Disease)')
axes[0].legend()
axes[0].set_xticks(range(1, 21))

# Plot 2: Confusion Matrix
cm_h = confusion_matrix(y_te_h, y_pred_h)
sns.heatmap(cm_h, annot=True, fmt='d', cmap='Purples', ax=axes[1],
            xticklabels=['Pred: Tidak','Pred: Sakit'],
            yticklabels=['Aktual: Tidak','Aktual: Sakit'])
axes[1].set_title('Confusion Matrix — KNN Heart Disease')

plt.suptitle(f'KNN (K={k_opt}) — Dataset Heart Disease UCI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Analisis KNN — Dataset Heart Disease

- **Dataset Heart Disease UCI** adalah dataset benchmark populer dengan 303 pasien dan 13 fitur medis.
- **Preprocessing:** StandardScaler digunakan untuk menormalisasi semua fitur karena KNN sangat sensitif terhadap perbedaan skala (contoh: kolesterol dalam ratusan vs. sex dalam 0/1).
- **Pemilihan K:** Dilakukan pencarian grid K=1 sampai K=20. K yang terlalu kecil menyebabkan model terlalu sensitif terhadap noise (overfitting), sedangkan K terlalu besar membuat batas keputusan terlalu halus (underfitting).
- **Confusion matrix** menunjukkan kemampuan model mendeteksi pasien sakit jantung. Dalam konteks klinis, **False Negative** (pasien sakit tapi diprediksi sehat) jauh lebih berbahaya daripada False Positive.
- **Perbandingan dengan model lain:** KNN relatif sederhana dan interpretatif, namun lambat saat prediksi pada dataset besar karena harus menghitung jarak ke semua data latih.
- **Kesimpulan:** KNN menunjukkan performa yang baik pada dataset Heart Disease. Untuk aplikasi klinis nyata, perlu dipertimbangkan trade-off antara precision dan recall untuk kelas positif (sakit jantung).

---
# 📌 RINGKASAN PERBANDINGAN MODEL

| Model | Tipe | Data | Fitur Utama | Metrik |
|---|---|---|---|---|
| Regresi Linear Sederhana | Regresi | Jam Belajar vs Nilai | 1 fitur | R², RMSE |
| Regresi Linear Berganda | Regresi | Harga Rumah | 3 fitur | R², RMSE |
| Regresi Logistik | Klasifikasi Biner | Diabetes | 3 fitur medis | Accuracy, F1 |
| KNN (data sendiri) | Klasifikasi Multi-kelas | Jenis Buah | 2 fitur | Accuracy, F1 |
| KNN (Kaggle) | Klasifikasi Biner | Heart Disease | 13 fitur medis | Accuracy, F1 |

## 💡 Kesimpulan Akhir

1. **Regresi Linear** cocok untuk memprediksi nilai kontinu (angka). Sederhana dan mudah diinterpretasikan, namun asumsi linearitas harus terpenuhi.

2. **Regresi Logistik** meskipun namanya 'regresi', sebenarnya digunakan untuk klasifikasi biner. Menggunakan fungsi sigmoid untuk mengubah output ke probabilitas.

3. **KNN** adalah algoritma non-parametrik yang intuitif — tidak membutuhkan asumsi distribusi data. Namun sensitif terhadap skala fitur dan nilai K yang dipilih.

4. **Normalisasi data** (StandardScaler) sangat penting untuk KNN dan Regresi Logistik agar model tidak bias terhadap fitur berskala besar.

5. **Pemilihan metrik** evaluasi harus disesuaikan dengan konteks: untuk deteksi penyakit, recall lebih penting dari accuracy; untuk prediksi harga, RMSE lebih relevan.